In [402]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [403]:
import pandas as pd
import numpy as np

In [404]:
path = "/content/drive/MyDrive/genre_dataset.csv"
df = pd.read_csv(path)

In [405]:
df.head(10)

,title,genre,overview
0,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...
1,The Dark Knight,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...
2,The Godfather: Part II,"Crime, Drama",The early life and career of Vito Corleone in ...
3,12 Angry Men,"Crime, Drama",A jury holdout attempts to prevent a miscarria...
4,The Lord of the Rings: The Return of the King,"Action, Adventure, Drama",Gandalf and Aragorn lead the World of Men agai...
5,Pulp Fiction,"Crime, Drama","The lives of two mob hitmen, a boxer, a gangst..."
6,Schindler's List,"Biography, Drama, History","In German-occupied Poland during World War II,..."
7,Inception,"Action, Adventure, Sci-Fi",A thief who steals corporate secrets through t...
8,The Lord of the Rings: The Fellowship of the Ring,"Action, Adventure, Drama",A meek Hobbit from the Shire and eight compani...
9,Forrest Gump,"Drama, Romance","The presidencies of Kennedy and Johnson, the e..."


In [406]:
def values_check(df):
  return df.isnull().sum(),df.duplicated().sum()

missing_values,duplicates = values_check(df)

In [407]:
df["genre"] = df["genre"].str.replace("Science Fiction", "Sci-Fi")
df = df[df["genre"] != ""]

In [408]:
remove_genres = [
    "News",
    "Game-Show",
    "Talk-Show",
    "Reality-TV",
    "Film-Noir",
    "Sport",
    "Short",
    "Biography",
    "Musical",
    "TV Movie",
    "Documentary",
    "Western",
    "Fantasy",
    "Mystery",
    "Music",
    "War",
    "History"

]

In [409]:
def remove_labels(genres):
    genres = genres.split(", ")
    genres = [g for g in genres if g not in remove_genres]
    return ", ".join(genres)

df["genre"] = df["genre"].apply(remove_labels)

In [410]:
df = df[df["genre"] != ""].reset_index(drop=True)

In [411]:
df["text"] = df["title"].fillna("") + " " + df["overview"].fillna("")

In [412]:
missing_values

,0
title,0
genre,0
overview,0


In [413]:
duplicates

np.int64(2730)

***There is no need to remove duplicates, as I have already removed them based on all three columns.***

In [414]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [415]:
X = df['text']

In [416]:
df["genre"] = df["genre"].str.split(", ")

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["genre"])
len(mlb.classes_)

11

In [417]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [418]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Bidirectional,LSTM,Embedding,Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [419]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")

In [420]:
tokenizer.fit_on_texts(X_train)

In [421]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [422]:
X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding='post', truncating="post")
X_test_pad = pad_sequences(X_test_seq ,maxlen=200, padding='post', truncating="post")

In [423]:
model = Sequential()

model.add(Embedding(input_dim=20000, output_dim=200))

model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.4))

model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.3))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(len(mlb.classes_), activation="sigmoid"))

In [424]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_12                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_13                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [425]:
optimizer = Adam(learning_rate=0.0004)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [426]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [427]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=25,
    batch_size=512,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 15s 234ms/step - accuracy: 0.1419 - loss: 0.5222 - val_accuracy: 0.2236 - val_loss: 0.4478
Epoch 2/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 224ms/step - accuracy: 0.2218 - loss: 0.4610 - val_accuracy: 0.2236 - val_loss: 0.4438
Epoch 3/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 229ms/step - accuracy: 0.2256 - loss: 0.4502 - val_accuracy: 0.2250 - val_loss: 0.4262
Epoch 4/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 237ms/step - accuracy: 0.2390 - loss: 0.4268 - val_accuracy: 0.2475 - val_loss: 0.4098
Epoch 5/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 241ms/step - accuracy: 0.2447 - loss: 0.4077 - val_accuracy: 0.2225 - val_loss: 0.4066
Epoch 6/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 237ms/step - accuracy: 0.2476 - loss: 0.3915 - val_accuracy: 0.2431 - val_loss: 0.3958
Epoch 7/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 230ms/step - accuracy: 0.2591 - loss: 0.3744 - val_accuracy: 0.2370 - val_loss: 0.3876
Epoch 8/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 240ms/step - accuracy: 0.2819 - loss: 0.3517 - val_accuracy: 0

In [428]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

186/186 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3357 - loss: 0.3705
Test Loss: 0.37049806118011475
Test Accuracy: 0.33568668365478516


In [429]:
y_pred = model.predict(X_test_pad)

y_pred = (y_pred >= 0.4).astype(int)

186/186 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step


In [430]:
from sklearn.metrics import multilabel_confusion_matrix

cm = multilabel_confusion_matrix(y_test, y_pred)

for i, genre in enumerate(mlb.classes_):
    print(f"\n{genre}")
    print(cm[i])


Action
[[3664  765]
 [ 654  866]]

Adventure
[[5077  242]
 [ 414  216]]

Animation
[[5577   36]
 [ 297   39]]

Comedy
[[2939 1236]
 [ 694 1080]]

Crime
[[4510  466]
 [ 614  359]]

Drama
[[1698 1167]
 [ 492 2592]]

Family
[[5449   13]
 [ 447   40]]

Horror
[[5103  113]
 [ 358  375]]

Romance
[[3758  832]
 [ 496  863]]

Sci-Fi
[[5304   96]
 [ 320  229]]

Thriller
[[4024  483]
 [ 712  730]]


In [431]:
# import matplotlib.pyplot as plt
# from sklearn.metrics import ConfusionMatrixDisplay, multilabel_confusion_matrix

# cms = multilabel_confusion_matrix(y_test, y_pred)

# for i, genre in enumerate(mlb.classes_):
#     disp = ConfusionMatrixDisplay(confusion_matrix=cms[i],
#                                   display_labels=["No", "Yes"])
#     disp.plot(cmap="Blues")
#     plt.title(genre)
#     plt.show()

In [432]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=mlb.classes_
))

              precision    recall  f1-score   support

      Action       0.53      0.57      0.55      1520
   Adventure       0.47      0.34      0.40       630
   Animation       0.52      0.12      0.19       336
      Comedy       0.47      0.61      0.53      1774
       Crime       0.44      0.37      0.40       973
       Drama       0.69      0.84      0.76      3084
      Family       0.75      0.08      0.15       487
      Horror       0.77      0.51      0.61       733
     Romance       0.51      0.64      0.57      1359
      Sci-Fi       0.70      0.42      0.52       549
    Thriller       0.60      0.51      0.55      1442

   micro avg       0.58      0.57      0.57     12887
   macro avg       0.59      0.45      0.47     12887
weighted avg       0.58      0.57      0.56     12887
 samples avg       0.54      0.57      0.53     12887



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
